# Software profesional en Acústica 2024-25 (M2i)

*This notebook contains a modification of the notebook [FEM_Helmholtz_equation_Robin](https://github.com/spatialaudio/computational_acoustics/blob/master/FEM_Helmholtz_equation_Robin.ipynb), created by Sascha Spors, Frank Schultz, Computational Acoustics Examples, 2018. The text/images are licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/). The code is released under the [MIT license](https://opensource.org/licenses/MIT).*

# Numerical Solution of the coupling two fluid domains by using an impedance boundary condition associated to a veil or local reacting rigid plate

This notebook illustrates the numerical solution of the wave equation for an harmonic excitation coupling a fluid domain with porous materials using the [Finite Element Method](https://en.wikipedia.org/wiki/Finite_element_method) (FEM). The method aims at an approximate solution by subdividing the area of interest into smaller parts with simpler geometry, linking these parts together and applying methods from the calculus of variations to solve the problem numerically. The FEM is a well established method for the numerical approximation of the solution of partial differential equations (PDEs). The solutions of PDEs are often known analytically only for rather simple geometries. FEM based simulations allow to gain insights into other more complex cases such this coupled fluid-porous problem.

First, we need to install NGSolve. In Google Colab or a fresh environment, run:

In [ ]:
try:
    import ngsolve
except ImportError:
    !pip install ngsolve

## Problem Statement

The inhomogeneous vector-valued Helmholtz equation governs the displacement field in the fluid domain $\Omega_{\mathrm{F}}$ and it is given as

\begin{equation}
\rho_{\mathrm{F}}c_{\mathrm{F}}^{2}\nabla\text{div}\, U(\mathbf{x}, \omega) + \omega^2\rho_{\mathrm{F}} U(\mathbf{x}, \omega) = - Q(\mathbf{x}, \omega) \qquad\text{in }\Omega_{\mathrm{F}}.
\end{equation}

The fluid domain is split in two parts separated by a impedance boundary condition located on the coupling interface $\Gamma_{\mathrm{I}}$. These coupling conditions between the porous and fluid domains is expressed in terms of the kinetic and kinematic boundary conditions,the continuity of the normal displacements on the coupling boundary and a jump condition on the pressure values, this is,

\begin{equation}
P(\mathbf{x}^{+}, \omega)-P(\mathbf{x}^{-}, \omega) = -i\omega Z_{\mathrm{S}}(\omega)\mathbf{U}(\mathbf{x}, \omega)\cdot\mathbf{n},\qquad \mathbf{x}\in\Gamma_{\mathrm{I}}.
\end{equation}

where $Z_{\mathrm{S}}(\omega)$ is the complex-valued surface impedance associated with the porous veil or the local-reacting rigid plate. In the case of the porous veil $Z_{\mathrm{S}}(\omega) = \alpha+i\beta/\omega$, where $\alpha$ and $\beta$ are constants. For the local-reacting rigid plate, $Z_{\mathrm{S}}(\omega) = (-\omega^2 m -i\omega r +s)/(-i\omega)$, where $m$, $r$, and $s$ are respectively the surface mass density, the damping and the stiffness associated with the rigid plate. The boundary conditions are completed assuming rigid boundaries on the rest of the boundary of the computational domain.

## Variational Formulation

Starting from the variational formulation of the Helmholtz equation (before application of Green's first theorem)

\begin{multline*}
\int_{\Omega_{\mathrm{F}}} \rho_{\mathrm{F}}c_{\mathrm{F}}^{2}\text{div}\, U(\mathbf{x}, \omega)\text{div}\, V(\mathbf{x}, \omega) \mathrm{d}x  - \omega^2\rho_{\mathrm{F}}\int_{\Omega_{\mathrm{F}}} U(\mathbf{x}, \omega)\cdot V(\mathbf{x}, \omega) \mathrm{d}x - i\omega Z_{\mathrm{S}}(\omega)\int_{\Gamma_{\mathrm{I}}} U(\mathbf{x}, \omega)\cdot\mathbf{n} V(\mathbf{x}, \omega)\cdot\mathbf{n} \mathrm{d}x
= -\int_{\Omega_{\mathrm{F}}} Q(\mathbf{x}, \omega) V(\mathbf{x}, \omega) \mathrm{d}x
\end{multline*}


It is common to express this integral equation in terms of the bilinear $a(P, V)$ and linear $L(V)$ forms 

\begin{equation}
a(U, V) = \int_{\Omega_{\mathrm{F}}} \rho_{\mathrm{F}}c_{\mathrm{F}}^{2}\text{div}\, U(\mathbf{x}, \omega)\text{div}\, V(\mathbf{x}, \omega) \mathrm{d}x  - \omega^2\rho_{\mathrm{F}}\int_{\Omega_{\mathrm{F}}} U(\mathbf{x}, \omega)\cdot V(\mathbf{x}, \omega) \mathrm{d}x - i\omega Z_{\mathrm{S}}(\omega)\int_{\Gamma_{\mathrm{I}}} U(\mathbf{x}, \omega)\cdot\mathbf{n} V(\mathbf{x}, \omega)\cdot\mathbf{n} \mathrm{d}x
\end{equation}

\begin{equation}
L(V) = -\int_{\Omega_{\mathrm{F}}} Q(\mathbf{x}, \omega) V(\mathbf{x}, \omega) \mathrm{d}x,
\end{equation}

where the variational problem to solve is stated as: Find $U$ such that

\begin{equation}
a(U, V) = L(V)\qquad \forall V.
\end{equation}

Splitting into real and imaginary parts $U = U_r + \mathrm{j} U_i$ and $V = V_r + \mathrm{j} V_i$, we get

\begin{multline*}
a_r = \int_{\Omega_{\mathrm{F}}} \left(\omega^2\rho_{\mathrm{F}} V_r(\mathbf{x}, \omega)\cdot U_r(\mathbf{x}, \omega) -  \omega^2\rho_{\mathrm{F}} V_i(\mathbf{x}, \omega)\cdot U_i(\mathbf{x}, \omega) - \rho_{\mathrm{F}}c_{\mathrm{F}}^{2}\text{div}\, U_r(\mathbf{x}, \omega) \text{div}\, V_r(\mathbf{x}, \omega) + \rho_{\mathrm{F}}c_{\mathrm{F}}^{2}\text{div}\, U_i(\mathbf{x}, \omega) \text{div}\, V_i(\mathbf{x}, \omega) \right) \mathrm{d}x\\
-i\omega Z_{\mathrm{S}}(\omega)\int_{\Gamma_{\mathrm{I}}} \left( V_i(\mathbf{x}, \omega)\cdot\mathbf{n} U_r(\mathbf{x}, \omega)\cdot\mathbf{n} + V_r(\mathbf{x}, \omega)\cdot\mathbf{n} U_i(\mathbf{x}, \omega)\cdot\mathbf{n} \right) \mathrm{d}x, 
\end{multline*}

\begin{multline*}
a_i = \int_{\Omega_{\mathrm{F}}} \left(\omega^2\rho_{\mathrm{F}} V_r(\mathbf{x}, \omega)\cdot U_i(\mathbf{x}, \omega) +  \omega^2\rho_{\mathrm{F}} V_i(\mathbf{x}, \omega)\cdot U_r(\mathbf{x}, \omega) - \rho_{\mathrm{F}}c_{\mathrm{F}}^{2}\text{div}\,U_i(\mathbf{x}, \omega) \text{div}\, V_r(\mathbf{x}, \omega) + \rho_{\mathrm{F}}c_{\mathrm{F}}^{2}\text{div}\,U_r(\mathbf{x}, \omega) \text{div}\, V_i(\mathbf{x}, \omega) \right) \mathrm{d}x\\ 
+i\omega Z_{\mathrm{S}}(\omega)\int_{\Gamma_{\mathrm{I}}} \left( V_r(\mathbf{x}, \omega)\cdot\mathbf{n} U_r(\mathbf{x}, \omega)\cdot\mathbf{n} - V_i(\mathbf{x}, \omega)\cdot\mathbf{n} U_i(\mathbf{x}, \omega)\cdot\mathbf{n} \right) \mathrm{d}s
\end{multline*}

for the bilinear form.

## Numerical Solution

The numerical solution of the variational problem is based on [NGSolve](https://ngsolve.org/), an open-source framework for numerical solution of PDEs.
Its high-level Python interface is used in the following to define the problem and compute the solution.
The implementation is based on the variational formulation derived above.
It is common in the FEM to denote the solution of the problem by $u$ and the test function by $v$.
The definition of the problem in NGSolve is very close to the mathematical formulation of the problem.

For the subsequent examples the solution of inhomogeneous wave equation for a point source $Q(\mathbf{x}) = \delta(\mathbf{x}-\mathbf{x_s})$ at position $\mathbf{x_s}$ is computed using the FEM.
A function is defined for this purpose, as well as for the plotting of the resulting sound field.

In [ ]:
import ngsolve as ng
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.geom2d import SplineGeometry
import numpy as np
import matplotlib.pyplot as plt


def fluid_impedance(mesh, frequency, xs, alpha=0., beta=0.):
    """
    Solve the vector Helmholtz equation for a displacement field in a fluid
    domain split by an impedance interface at x=3.25.

    Parameters
    ----------
    mesh      : NGSolve Mesh
    frequency : float, excitation frequency in Hz
    xs        : (x, y) tuple/list with the point-source location
    alpha     : float, real part of surface impedance
    beta      : float, coefficient for imaginary part (Zs = alpha + 1j*beta/omega)

    Returns
    -------
    sol_re, sol_im : GridFunction components (real and imaginary displacement fields)
    """

    # Physical parameters
    omega_val = 2 * np.pi * frequency
    rhoF  = 1.21           # fluid mass density [kg/m^3]
    bulkF = rhoF * 343**2  # bulk modulus [Pa]
    Zs    = alpha + 1j * beta / omega_val  # complex surface impedance
    Zs_re = float(Zs.real)
    Zs_im = float(Zs.imag)

    # ----------------------------------------------------------------
    # Function space: Raviart-Thomas for the displacement vector field
    # Real and imaginary parts live in a product space RT x RT
    # ----------------------------------------------------------------
    RT  = HDiv(mesh, order=1, dirichlet="outer")
    fes = RT * RT   # product space (u_r, u_i)

    (u_r, u_i), (v_r, v_i) = fes.TnT()

    # Normal to the coupling interface (used in jump terms)
    n = specialcf.normal(mesh.dim)

    # ----------------------------------------------------------------
    # Bilinear form (real-part and imaginary-part equations)
    # ----------------------------------------------------------------
    a = BilinearForm(fes)

    # Volumetric terms — real-part equation
    a += ( omega_val**2 * rhoF * InnerProduct(u_r, v_r)
          -omega_val**2 * rhoF * InnerProduct(u_i, v_i)
          -bulkF * div(u_r) * div(v_r)
          +bulkF * div(u_i) * div(v_i) ) * dx

    # Volumetric terms — imaginary-part equation
    a += ( omega_val**2 * rhoF * InnerProduct(u_r, v_i)
          +omega_val**2 * rhoF * InnerProduct(u_i, v_r)
          -bulkF * div(u_i) * div(v_r)
          -bulkF * div(u_r) * div(v_i) ) * dx   # note: sign fixed vs original

    # Interior-facet (jump) terms from the impedance interface
    # dS("coupling") integrates over the marked interior facets
    # u('+')·n('+') is the normal trace on one side of the interface
    if abs(Zs_im) > 0:
        a += ( omega_val * Zs_im * (u_r('+') * n('+')) * (v_i('+') * n('+'))
              +omega_val * Zs_im * (u_i('+') * n('+')) * (v_r('+') * n('+')) ) * dx(skeleton=True, definedon=mesh.Boundaries(""))
        # For interior skeleton integration use the 'coupling' region defined below

    if abs(Zs_re) > 0:
        a += ( omega_val * Zs_re * (u_r('+') * n('+')) * (v_r('+') * n('+'))
              -omega_val * Zs_re * (u_i('+') * n('+')) * (v_i('+') * n('+')) ) * dx(skeleton=True, definedon=mesh.Boundaries(""))

    # ----------------------------------------------------------------
    # Linear form (zero volumetric source — point source added later)
    # ----------------------------------------------------------------
    f = LinearForm(fes)
    f += InnerProduct(CoefficientFunction((0, 0)), v_r) * dx
    f += InnerProduct(CoefficientFunction((0, 0)), v_i) * dx

    # Assemble
    a.Assemble()
    f.Assemble()

    # ----------------------------------------------------------------
    # Point source: add -delta(x - xs) to the real-part RHS
    # NGSolve does not have a built-in PointSource; we approximate it
    # by setting a very concentrated Gaussian or by direct dof loading.
    # Here we use the "PointSource" approach via SetPointSource.
    # ----------------------------------------------------------------
    # In NGSolve, point sources for vector spaces are handled by
    # directly modifying the rhs vector at the closest DOF.
    # A simpler and robust approach is to use a sharp Gaussian.
    eps_src = 0.05  # width of the Gaussian approximation
    xs_x, xs_y = xs
    point_source = -1.0 / (np.pi * eps_src**2) * exp(
        -((x - xs_x)**2 + (y - xs_y)**2) / eps_src**2
    )
    f_ps = LinearForm(fes)
    f_ps += InnerProduct(CoefficientFunction((point_source, 0)), v_r) * dx
    f_ps += InnerProduct(CoefficientFunction((0, 0)), v_i) * dx
    f_ps.Assemble()

    # ----------------------------------------------------------------
    # Solve
    # ----------------------------------------------------------------
    sol = GridFunction(fes)
    sol.vec.data = a.mat.Inverse(fes.FreeDofs()) * f_ps.vec

    sol_re = sol.components[0]
    sol_im = sol.components[1]

    return sol_re, sol_im


def plot_soundfield(u, title=r'$P(\mathbf{x}, \omega)$'):
    """Plots magnitude of a vector field solution using matplotlib."""
    # Draw in the NGSolve webgui
    Draw(u)
    print(f"Plotting: {title}")

### Sound Field transmitted through a layer of porous material

Two 2D-dimensional rooms are connected by a layer of porous material. The porous layer decrease the amplitude of the pressure field and the sound power level is computed on both rooms to check the efficiency absorbing the incident pressure generated by a point source in the left room

In [ ]:
from netgen.geom2d import SplineGeometry

# Build geometry: two rectangular rooms connected through a narrow channel at x=3.25
# Left room:  [0,3] x [0,4]  +  channel [3, 3.25] x [1.5, 2.5]
# Right room: [3.5,6] x [0,4]  +  channel [3.25, 3.5] x [1.5, 2.5]

geo = SplineGeometry()

# Define all corner points
# Left room outer boundary (counter-clockwise)
pts = [
    geo.AppendPoint(0,   0),    # 0
    geo.AppendPoint(3,   0),    # 1
    geo.AppendPoint(3,   1.5),  # 2
    geo.AppendPoint(3.25,1.5),  # 3
    geo.AppendPoint(3.25,2.5),  # 4
    geo.AppendPoint(3,   2.5),  # 5
    geo.AppendPoint(3,   4),    # 6
    geo.AppendPoint(0,   4),    # 7
    geo.AppendPoint(3.5, 1.5),  # 8
    geo.AppendPoint(6,   1.5),  # 9
    geo.AppendPoint(6,   0),    # 10
    geo.AppendPoint(6,   2.5),  # 11
    geo.AppendPoint(3.5, 2.5),  # 12
    geo.AppendPoint(6,   4),    # 13
    geo.AppendPoint(3.5, 4),    # 14
    geo.AppendPoint(3.5, 0),    # 15
]

# Left room boundary (material 1 = input room)
geo.Append(["line", pts[0],  pts[1]],  leftdomain=1, rightdomain=0, bc="outer")
geo.Append(["line", pts[1],  pts[2]],  leftdomain=1, rightdomain=0, bc="outer")
geo.Append(["line", pts[2],  pts[3]],  leftdomain=1, rightdomain=0, bc="outer")
geo.Append(["line", pts[3],  pts[4]],  leftdomain=1, rightdomain=2, bc="coupling")  # impedance interface
geo.Append(["line", pts[4],  pts[5]],  leftdomain=1, rightdomain=0, bc="outer")
geo.Append(["line", pts[5],  pts[6]],  leftdomain=1, rightdomain=0, bc="outer")
geo.Append(["line", pts[6],  pts[7]],  leftdomain=1, rightdomain=0, bc="outer")
geo.Append(["line", pts[7],  pts[0]],  leftdomain=1, rightdomain=0, bc="outer")

# Right room boundary (material 2 = output room)
geo.Append(["line", pts[15], pts[10]], leftdomain=2, rightdomain=0, bc="outer")
geo.Append(["line", pts[10], pts[9]],  leftdomain=2, rightdomain=0, bc="outer")
geo.Append(["line", pts[9],  pts[8]],  leftdomain=2, rightdomain=0, bc="outer")
geo.Append(["line", pts[8],  pts[15]], leftdomain=2, rightdomain=0, bc="outer")
geo.Append(["line", pts[11], pts[13]], leftdomain=2, rightdomain=0, bc="outer")
geo.Append(["line", pts[13], pts[14]], leftdomain=2, rightdomain=0, bc="outer")
geo.Append(["line", pts[14], pts[12]], leftdomain=2, rightdomain=0, bc="outer")
geo.Append(["line", pts[12], pts[11]], leftdomain=2, rightdomain=0, bc="outer")

geo.SetMaterial(1, "input")
geo.SetMaterial(2, "output")

ngmesh = geo.GenerateMesh(maxh=0.2)
mesh = Mesh(ngmesh)

# Plot mesh
Draw(mesh)
print("Mesh created with", mesh.nv, "vertices")

The two-dimensional sound field in a rectangular room (rectangular plate) with homogeneous Robin boundary conditions is computed for the frequency $f=120$ Hz and source position $x_s = (1.2,3.2)$ m.

In [ ]:
f = 120.0        # frequency [Hz]
xs = (1.2, 3.2)  # source position

# Compute solution without surface impedance (alpha=0, beta=0)
sol_re, sol_im = fluid_impedance(mesh, f, xs, alpha=0., beta=0.)

# Plot displacement field: real part
Draw(sol_re, mesh, "Re(U)")

# Plot displacement magnitude
mag = sqrt(sol_re[0]**2 + sol_re[1]**2 + sol_im[0]**2 + sol_im[1]**2)
Draw(mag, mesh, "|U|")

Compute the sound velocity level (SVL) in the output and input room

In [ ]:
v_ref = 5e-8  # reference velocity [m/s]
omega_val = 2.0 * np.pi * f

mag = sqrt(sol_re[0]**2 + sol_re[1]**2 + sol_im[0]**2 + sol_im[1]**2)

# Integrate over each subdomain using domain labels
SVL_input  = 20 * np.log10(omega_val * Integrate(mag, mesh, definedon=mesh.Materials("input"))  / v_ref)
SVL_output = 20 * np.log10(omega_val * Integrate(mag, mesh, definedon=mesh.Materials("output")) / v_ref)

print('SVL in the input room without surface impedance:', SVL_input,  'dB')
print('SVL in the output room without surface impedance:', SVL_output, 'dB')

In [ ]:
f = 120.0        # frequency [Hz]
xs = (1.2, 3.2)  # source position

# Compute solution with surface impedance (alpha=1e3, beta=5e2)
sol_re, sol_im = fluid_impedance(mesh, f, xs, alpha=1e3, beta=5e2)

# Plot displacement field: real part
Draw(sol_re, mesh, "Re(U) with impedance")

# Plot displacement magnitude
mag = sqrt(sol_re[0]**2 + sol_re[1]**2 + sol_im[0]**2 + sol_im[1]**2)
Draw(mag, mesh, "|U| with impedance")

Compute the SVL values associated with the input and output room, and the insertion loss (IL) values 

In [ ]:
v_ref = 5e-8  # reference velocity [m/s]

mag = sqrt(sol_re[0]**2 + sol_re[1]**2 + sol_im[0]**2 + sol_im[1]**2)

SVL_input_impedance  = 10 * np.log10(omega_val * Integrate(mag, mesh, definedon=mesh.Materials("input"))  / v_ref)
SVL_output_impedance = 10 * np.log10(omega_val * Integrate(mag, mesh, definedon=mesh.Materials("output")) / v_ref)

print('SVL in the input room with surface impedance:',  SVL_input_impedance,  'dB')
print('SVL in the output room with surface impedance:', SVL_output_impedance, 'dB')

# Insertion loss for the particle velocity
IL = SVL_output - SVL_output_impedance
print('Insertion loss (IL):', IL, 'dB')

**Copyright**

This notebook is provided as [Open Educational Resource](https://en.wikipedia.org/wiki/Open_educational_resources). Feel free to use the notebook for your own purposes. The text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/), the code of the IPython examples under the [MIT license](https://opensource.org/licenses/MIT).